In [18]:
import pandas as pd
import csv
from pathlib import Path

BASE = Path("/Users/apple/Desktop/Annotate_Files")
N = 180
ANNOTATORS = ("MoAli", "Asma", "Nika")

def read_smart(path):
    try:
        df = pd.read_csv(path, sep="\t", encoding="utf-8-sig")
        if df.shape[1] > 1:
            return df
    except Exception:
        pass
    return pd.read_csv(path, encoding="utf-8-sig")

def prepare_stt_csv(input_file, output_file, n=N):
    df = read_smart(input_file)

    
    df.columns = [str(c).strip().lower().lstrip("\ufeff") for c in df.columns]

    
    df = df.head(n).copy()

    
    if "line" in df.columns:
        df = df.rename(columns={"line": "Line"})
    else:
        df.insert(0, "Line", range(1, len(df) + 1))

    # Sentence
    sent_col = None
    for cand in ("text", "sentence", "transcript", "utterance", "content"):
        if cand in df.columns:
            sent_col = cand
            break
    if sent_col is None:
        sent_col = [c for c in df.columns if c != "Line"][0]
    df = df.rename(columns={sent_col: "Sentence"})

    
    df["S"] = 0
    df["I"] = 0
    df["D"] = 0

    
    df["T"] = df["Sentence"].fillna("").astype(str).str.split().str.len()

    
    df["Checked"] = ""
    df["Annotator"] = ""
    df.loc[0:59,   "Annotator"] = ANNOTATORS[0]
    df.loc[60:119, "Annotator"] = ANNOTATORS[1]
    df.loc[120:179,"Annotator"] = ANNOTATORS[2]

    
    df = df[["Line", "Sentence", "S", "I", "D", "T", "Checked", "Annotator"]]

    
    df.to_csv(output_file, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
    print(f"Saved -> {output_file}")

#(AssemblyAI)
prepare_stt_csv(BASE / "metri_6_assemblyai.csv", BASE / "STT_metri_6_Assembly.csv")


Saved -> /Users/apple/Desktop/Annotate_Files/STT_metri_6_Assembly.csv


In [21]:
import pandas as pd
import csv
from pathlib import Path

BASE = Path("/Users/apple/Desktop/Annotate_Files")
N = 180
ANNOTATORS = ("MoAli", "Asma", "Nika")

def read_table(path: Path) -> pd.DataFrame:
    for enc in ("utf-8-sig", "utf-8", "cp1256", "latin1"):
        try:
            return pd.read_csv(path, sep=None, engine="python", encoding=enc)
        except UnicodeDecodeError:
            continue
        except Exception:
            continue
    return pd.read_csv(path, sep=None, engine="python", encoding="latin1")

def prepare_whisper(input_file: Path, output_file: Path, n: int = N):
    df = read_table(input_file)

    df.columns = [str(c).strip().lower().lstrip("\ufeff") for c in df.columns]

    line_col = "line" if "line" in df.columns else None
    if "text_fa" in df.columns:
        text_col = "text_fa"
    elif "text_en" in df.columns:
        text_col = "text_en"
    elif "text" in df.columns:
        text_col = "text"
    else:
        text_col = next(c for c in df.columns if c != "line")

    take_cols = [c for c in ["line", text_col] if c in df.columns]
    df = df[take_cols].head(n).copy()

    if "line" in df.columns:
        df = df.rename(columns={"line": "Line"})
    else:
        df.insert(0, "Line", range(1, len(df) + 1))

    df = df.rename(columns={text_col: "Sentence"})

    df["S"] = 0
    df["I"] = 0
    df["D"] = 0

    df["T"] = df["Sentence"].fillna("").astype(str).str.split().str.len()

    df["Checked"] = ""
    df["Annotator"] = ""
    df.loc[0:59,   "Annotator"] = ANNOTATORS[0]
    df.loc[60:119, "Annotator"] = ANNOTATORS[1]
    df.loc[120:179,"Annotator"] = ANNOTATORS[2]

    df = df[["Line", "Sentence", "S", "I", "D", "T", "Checked", "Annotator"]]

    df.to_csv(output_file, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
    print(f"Saved -> {output_file}")

# Whisper (metri_6)
prepare_whisper(
    BASE / "metri_6_whisper.csv",
    BASE / "STT_metri_6_Whisper.csv"
)


Saved -> /Users/apple/Desktop/Annotate_Files/STT_metri_6_Whisper.csv
